<a href="https://colab.research.google.com/github/alfredojavier55/wid-Global_index/blob/main/executorch_pip_package_XNNPACK_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook uses the prebuilt `executorch` pip package to export and run an XNNPACK-delegated model. This shows that it's possible to do so without needing to clone the repo and install from source, simplifying the steps currently at https://pytorch.org/executorch/stable/getting-started-setup.html.

Note that this will only work if the model is compatible with the operators and backends linked into the `executorch>=0.2.0` package: i.e., it uses the core ATen operator set, and may use the XNNPACK backend. But this example doesn't support custom operators or other backends.

Install the prebuilt executorch pip package.

NOTE: You may see the message ERROR: pip's dependency resolver ... for packages like torchaudio and torchtext, but it won't affect this demo.

In [ ]:
!pip install executorch==0.6.0

# Instead of 0.6.0, use the latest version in https://pypi.org/project/executorch/#history

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 M

Demonstrate that the packages are imported successfully.

In [ ]:
from executorch import version

version.__version__

'0.6.0+cpu'

In [ ]:
import torch

torch.__version__

# executorch pip package uses torch as dependency

'2.7.0+cu126'

Demonstrate that the native pybindings module imports successfully, and provides basic operators.

Pybindings are great for doing testing and prototyping. Ultimately, you'll need to write the executorch runtime integration in C++ during productionization.

In [ ]:
from executorch.runtime import Runtime

runtime = Runtime.get()

operator_names = runtime.operator_registry.operator_names

print(len(operator_names))

print(next(iter(operator_names)))

217
aten::any.dims_out


Let's do inference on MobileNetV2.

In order to do that,
- first we need to find the nn.Module
- use `torch.export()` to export the graph
- And subsequently, generate `.pte` file
- Lastly, do inference via executorch runtime on the `.pte` file

In [ ]:
from torchvision.models import mobilenet_v2
from torchvision.models.mobilenetv2 import MobileNet_V2_Weights

mv2 = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT) # This is torch.nn.Module

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 64.9MB/s]


In [ ]:
from executorch.exir import to_edge
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

model = mv2.eval() # turn into evaluation mode

example_inputs = (torch.randn((1, 3, 224, 224)),) # Necessary for exporting the model

exported_graph = torch.export.export(model, example_inputs) # Core Aten graph

edge = to_edge(exported_graph) # Edge Dialect

edge_delegated = edge.to_backend(XnnpackPartitioner()) # Parts of the graph are delegated to XNNPACK

executorch_program = edge_delegated.to_executorch() # ExecuTorch program

In [ ]:
pte_path = "mv2_xnnpack.pte"

with open(pte_path, "wb") as file:
    executorch_program.write_to_file(file) # Serializing into .pte file

Try loading and executing the XNNPACK-delegated model using the executorch pip package.

In [ ]:
program = runtime.load_program(pte_path)
method = program.load_method("forward")

[program.cpp:135] InternalConsistency verification requested but not available


In [ ]:
t = torch.randn((1, 3, 224, 224))

output = method.execute([t])
assert len(output) == 1, f"Unexpected output length {len(output)}"
assert output[0].size() == torch.Size([1, 1000]), f"Unexpected output size {output[0].size()}"
print("PASS")

PASS
